# Debate - 선생님과 제자

In [1]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
MODEL_API_VERSION=os.getenv('MODEL_API_VERSION')
if MODEL_API_VERSION:
    os.environ['MODEL_API_VERSION'] = MODEL_API_VERSION
if AZURE_OPENAI_API_KEY:
    print(AZURE_OPENAI_API_KEY[:10])
else:
    print("AZURE_OPENAI_API_KEY가 설정되지 않았습니다.")
print(MODEL_NAME, MODEL_API_VERSION)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

langsmith_key = os.getenv('LANGSMITH_API_KEY')
if langsmith_key:
    os.environ['LANGCHAIN_API_KEY'] = langsmith_key
ep = os.getenv('LANGCHAIN_ENDPOINT')
if ep:
    os.environ['LANGCHAIN_ENDPOINT'] = ep
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    _lk = os.getenv('LANGSMITH_API_KEY')
    if _lk and len(_lk) > 0:
        print('랭스미스로 추적 중입니다 :', _lk[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
gpt-5-nano 2025-01-01-preview


In [2]:
llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version=os.environ.get('MODEL_API_VERSION', '2024-12-01-preview'),
    model = MODEL_NAME,
)

In [3]:
from typing import TypedDict, Literal, Annotated, List
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.graph.message import add_messages

def window_clip_message(old, new):
    return (old + new)[-4:]

class S(TypedDict, total=False):
    user_input: str
    topic: str
    student_level: str
    max_turns: int
    turn: int
    understood: bool
    final_report: str
    messages: Annotated[List[BaseMessage], window_clip_message]

MAX_TURNS = 4

In [5]:
def init_node(state: S) -> S:

    prompt = ChatPromptTemplate.from_template(
        "당신은 유저의 입력으로 부터 토픽을 추출하는 도우미입니다."
        "유저 질문의 의도를 파악해 설명을 하기에 적절한 주제의 문장으로 수정하세요. "
        "유저 질문 : {user_input}"        
    )
    
    chain = prompt | llm | StrOutputParser()
    out = chain.invoke({"user_input": state["user_input"]})
    return {
        "topic": out,
        "turn": 0,
        "understood": False,
    }


In [6]:
TEACHER_SYS = """당신은 친절하고 훌륭한 선생님입니다.
학생과 대화한 내용을 바탕으로 학생의 수준에 맞게 주제를 설명합니다.

# 주제: {topic}
# 학생 수준: {student_level}

# 규칙:
- 이번 턴에는 주제의 한 부분만 간결하게 설명하세요(최대 6문장).
- 쉬운 비유나 예시가 필요하다면 포함하세요.
- 마지막에 한 줄로 “이 부분 이해되나요?”와 같이 확인 질문을 붙이세요.
- 학생이 이해 못 한 부분만 보완하세요. 새 주제는 열지 마세요.

# 대화 히스토리:
{messages}

"""

def teacher_node(state: S) -> S:
    prompt = ChatPromptTemplate.from_template(TEACHER_SYS)
    chain = prompt | llm | StrOutputParser()
    out = chain.invoke({"topic": state["topic"], "messages": state["messages"], "student_level": state["student_level"]})
    message = f"##[선생님 설명]: {out}"
    return {"messages": [message]}  

In [7]:
STUDENT_SYS = """당신은 성실한 제자입니다. 주제에 대해 선생님이 설명하는 내용을 이해하고 질문하세요.

# 주제: {topic}
# 당신의 수준 : {student_level}

규칙:
- 바로 직전 선생님의 설명 중 가장 헷갈리는 단 하나의 포인트만 짚어 질문하세요(최대 2문장).
- 모든 것이 어느정도 이해되었다면 더이상 질문하지 말고 정확히 다음 한 단어만 출력하세요: "understood"
(추가 문장/문장부호 금지).

선생님 설명:
{messages}
"""

def student_node(state: S) -> S:
    prompt = ChatPromptTemplate.from_template(STUDENT_SYS)
    chain = prompt | llm | StrOutputParser()
    out = chain.invoke({"topic": state["topic"], "messages": state["messages"][-1], "student_level": state["student_level"]})
    understood = "understood" in out
    
    message = f"##[제자 질문]: {out}"

    return {
        "messages": [message],
        "turn": state["turn"] + 1,
        "understood": understood,
    }


In [9]:
JUDGE_SYS = """당신은 토픽에 대한 선생님의 설명을 정리하는 최종 작성자입니다.
대화 전체를 참고하여 아래 형식의 최종 보고서를 작성하세요.
1) 핵심 요점 3~5개 (불릿)
2) 쉬운 비유나 예시가 있다면 포함
3) 어려울 수 있는 개념이나 헷갈릴 수 있는 부분을 정리

# 주제: {topic}

# 대화 히스토리:
{messages}
"""

def judge_node(state: S) -> S:
    prompt = ChatPromptTemplate.from_template(JUDGE_SYS)
    chain = prompt | llm | StrOutputParser()
    out = chain.invoke({"topic": state["topic"], "messages": state["messages"]})
    return {"final_report": out}

In [10]:
def choose_after_student(state: S) -> Literal["to_teacher", "to_end"]:
    # 제자가 "이해했습니다"면 종료, 아니면 다시 교사에게
    turn = int(state.get("turn", 0))
    max_turns = int(state.get("max_turns", MAX_TURNS))
    if state.get("understood", False):
        return "to_judge"
    return "to_judge" if turn >= max_turns else "to_teacher"

In [11]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(
    S,
    input_keys=["user_input", "max_turns"],
    output_keys=["final_report"],
)

g.add_node("init", init_node)
g.add_node("teacher", teacher_node)
g.add_node("student", student_node)
g.add_node("judge", judge_node)

g.add_edge(START, "init")
g.add_edge("init", "teacher")
g.add_edge("teacher", "student")

g.add_conditional_edges(
    "student",
    choose_after_student,
    {"to_teacher": "teacher", "to_judge": "judge"},
)

g.add_edge("judge", END)

from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
graph = g.compile(checkpointer=checkpointer)

In [12]:
cfg = {"configurable": {"thread_id": "lesson-1"}}
out = graph.stream({"user_input": "생성형 AI(LLM)를 사용한 AI Agent 개념", "max_turns": 4, "student_level": "중학생"}, config=cfg)
for state in out:
    print(state)
    
print('최종 보고서 =========================')
print(state['judge']['final_report'])

{'init': {'topic': '다음과 같이 주제 문장으로 수정해 보았습니다. 필요에 따라 가장 적합한 versión를 선택해 사용하세요.\n\n- 간단한 버전: 생성형 AI(LLM)를 활용한 AI 에이전트의 개념과 핵심 원리를 소개한다.\n- 중간 버전: 생성형 AI(LLM) 기반 AI 에이전트의 정의, 구성 요소, 작동 원리 및 대표 활용 사례를 다룬다.\n- 심층 버전: 생성형 AI(LLM)를 토대로 한 AI 에이전트의 설계 원칙과 구현 방법, 도전 과제 및 미래 방향을 설명한다.', 'turn': 0, 'understood': False}}
{'teacher': {'messages': ["##[선생님 설명]: 생성형 AI(LLM)를 활용한 AI 에이전트는 사람의 지시를 이해하고 필요한 정보를 찾아 대답이나 작업을 수행하는 똑똑한 소프트웨어 도우미예요.\nLLM은 많은 글과 대화를 학습해 문장을 만들어 내는 '뇌' 같은 역할을 하며, 에이전트의 대화를 가능하게 합니다.\n에이전트는 목표를 정하고 그것을 달성하기 위한 최선의 행동을 선택하며, 필요한 도구를 함께 관리합니다.\n예를 들어 숙제를 도와달라면 질문을 이해하고 관련 정보를 찾아 답을 구성해 주는 식으로 작동합니다.\n하지만 모든 답이 완전하진 않아 사람의 확인이 필요할 때가 있습니다.\n이 부분 이해되나요?"]}}
{'student': {'messages': ["##[제자 질문]: 에이전트가 목표를 정하고 최선의 행동을 선택한다는 부분이 헷갈립니다. 최선의 행동은 어떤 원리로 결정되며, 에이전트가 '도구를 함께 관리'하는 구체적인 기준은 무엇인가요?"], 'turn': 1, 'understood': False}}
{'teacher': {'messages': ['##[선생님 설명]: 에이전트가 목표를 정하고 최선의 행동을 선택하는 원리는 "목표를 달성하기 위한 가장 도움이 되는 다음 한 걸음을 고르는 규칙"이에요.\n예를 들어 숙제를 도와줄 때는 먼저 무엇이 필요한지 파악하고, 여러 가능한 방법

In [14]:
cfg = {"configurable": {"thread_id": "lesson-1"}}
out = graph.stream({"user_input": "생성형 AI(LLM)를 사용한 AI Agent 개념", "max_turns": 4, "student_level": "ai 전문가"}, config=cfg)
for state in out:
    print(state)
    
print('최종 보고서 =========================')
print(state['judge']['final_report'])

{'init': {'topic': '다음은 유저의 질문 의도를 반영한 적절한 주제 문장(주제 문구) 예시들입니다. 필요에 따라 하나를 선택하거나 수정해 사용하세요.\n\n- 권장 주제 문장: "생성형 AI(LLM)를 활용한 AI 에이전트의 정의, 구성 요소, 작동 원리 및 실제 활용 사례를 설명한다."\n- 옵션 1: "생성형 AI(LLM)를 이용한 AI 에이전트의 개념과 기본 설계 원리를 설명한다."\n- 옵션 2: "생성형 AI를 활용한 에이전트의 정의, 아키텍처, 인터랙션 패턴 및 응용 사례를 다룬다."\n- 옵션 3: "생성형 AI 기반 에이전트의 구현 방법과 기술적 도전 과제를 중심으로 설명한다."\n- 옵션 4: "생성형 AI(Large Language Model)를 활용한 AI 에이전트의 이론적 프레임워크와 평가 방법을 탐구한다."\n\n추가로 다루고 싶은 범위나 대상 독자(초보/전문가)에 맞춰 문장을 더 구체화해 드릴 수 있습니다. 어떤 톤이나 깊이가 좋으신가요?', 'turn': 0, 'understood': False}}
{'teacher': {'messages': ['##[선생님 설명]: 데이터 기반으로 가중치를 학습할 때는 목표 성과를 예측하는 점수 s = w·x를 실제 관측치 y와 일치시키는 손실 함수를 사용합니다. 대표적으로 예측 오차를 최소화하는 회귀 손실인 평균제곱오차(MSE)와 평균절대오차(MAE), 이상치에 덜 민감한 허버(Huber) 손실이 많이 쓰입니다. 만약 목표가 후보 간 순서를 정확히 매기는 것이라면 쌍(pair) 손실(예: hinge, RankNet)이나 리스트 기반의 리스트뉴(ListNet/ListMLE) 같은 랭킹 손실을 사용합니다. 가중치의 일반화와 과적합 방지를 위해 L2 같은 정규화도 함께 적용합니다. 평가 지표로는 예측 정확도(MSE/MAE/RMSE, R^2), 순위 상관관계(Spearman/Kendall), 그리고 실제 비즈니스 지표의 개선 같은 KPIs를 함께 봅니다. 현업에서는 목표가 정렬이라면 랭킹 